# Code-to-Code Comparison: Dinwoodie

### National Renewable Energy Laboratory

#### Rob Hammond

##### 29 March 2022

In [1]:
from copy import deepcopy
from time import perf_counter
from pprint import pprint

import yaml
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from wombat.core import Simulation
from wombat.core.library import DINWOODIE, load_yaml

pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 1000)
pd.options.display.float_format = '{:,.2f}'.format
%matplotlib inline

In [2]:
# Converting Labor values to fixed cost input for the base case
tech_salary_annual = 80000
techs = 20
capacity = 240 * 1000  # 240 -> kW
f"{tech_salary_annual * techs / capacity:.4f}"


'6.6667'

In [3]:
configs = [
    "Ocean_Wind_1_100pct_reduction-90",
    "CVOW-C_100pct_reduction-90",
    #"more_techs_100pct_reduction",
    "fewer_techs_100pct_reduction"
    #"failure_50_100pct_reduction",
    #"failure_200_100pct_reduction",
    #"no_hlvs_100pct_reduction",
    #"no_weather_100pct_reduction",
    #"historic_weather_100pct_reduction",
    #"manual_resets_only_100pct_reduction",
    #"minor_repairs_only_100pct_reduction",
    #"medium_repairs_only_100pct_reduction",
    #"major_repairs_only_100pct_reduction",
    #"major_replacements_only_100pct_reduction",
    #"annual_service_only_100pct_reduction",
]
columns = deepcopy(configs)
results = {
    "availability - time based": [],
    "availability - production based": [],
    "capacity factor - net": [],
    "capacity factor - gross": [],
    "power production": [],
    "task completion rate": [],
    "annual direct O&M cost": [],
    "annual vessel cost": [],
    "ctv cost": [],
    "fsv cost": [],
    "hlv cost": [],
    "annual repair cost": [],
    "annual technician cost": [],
    "ctv utilization": [],
    "fsv utilization": [],
    "hlv utilization": [],
    "project capacity": []
}


        

In [4]:
for config in configs:
    # Run the simulation
    start = perf_counter()
    sim = Simulation(DINWOODIE, f"{config}.yaml")
    sim.run()
    end = perf_counter()
    print(f"{sim.config.name.rjust(50)} | {(end - start) / 60:.2f} m")
    
    # Gather the results of interest
    years = sim.metrics.events.year.unique().shape[0]
    mil = 1000000
    
    availability_time = sim.metrics.time_based_availability(frequency="project", by="windfarm")
    availability_production = sim.metrics.production_based_availability(frequency="project", by="windfarm")
    cf_net = sim.metrics.capacity_factor(which="net", frequency="project", by="windfarm")
    cf_gross = sim.metrics.capacity_factor(which="gross", frequency="project", by="windfarm")
    power_production = sim.metrics.power_production(frequency="project", by_turbine=False).values[0][0]
    completion_rate = sim.metrics.task_completion_rate(which="both", frequency="project")
    parts = sim.metrics.events[["materials_cost"]].sum().sum()
    techs = sim.metrics.project_fixed_costs(frequency="project", resolution="low").operations[0]
    total = sim.metrics.events[["total_cost"]].sum().sum()
    capacity = sim.metrics.project_capacity
    
    equipment = sim.metrics.equipment_costs(frequency="project", by_equipment=True)
    equipment_sum = equipment.sum().sum()
    hlv = equipment[[el for el in equipment.columns if "Heavy Lift Vessel" in el]].sum().sum()
    fsv = equipment[[el for el in equipment.columns if "Field Support Vessel" in el]].sum().sum()
    ctv = equipment[[el for el in equipment.columns if "Crew Transfer Vessel" in el]].sum().sum()
    
    utilization = sim.metrics.service_equipment_utilization(frequency="project")
    hlv_ur = utilization[[el for el in utilization.columns if "Heavy Lift Vessel" in el]].mean().mean()
    fsv_ur = utilization[[el for el in utilization.columns if "Field Support Vessel" in el]].mean().mean()
    ctv_ur = utilization[[el for el in utilization.columns if "Crew Transfer Vessel" in el]].mean().mean()
    
    # Log the results of interest
    results["availability - time based"].append(availability_time)
    results["availability - production based"].append(availability_production)
    results["capacity factor - net"].append(cf_net)
    results["capacity factor - gross"].append(cf_gross)
    results["power production"].append(power_production)
    results["task completion rate"].append(completion_rate)
    results["annual direct O&M cost"].append((total + techs) / mil / years)
    results["annual vessel cost"].append(equipment_sum / mil / years)
    results["ctv cost"].append(ctv / mil / years)
    results["fsv cost"].append(fsv / mil / years)
    results["hlv cost"].append(hlv / mil / years)
    results["annual repair cost"].append(parts / mil / years)
    results["annual technician cost"].append(techs / mil / years)
    results["ctv utilization"].append(ctv_ur)
    results["fsv utilization"].append(fsv_ur)
    results["hlv utilization"].append(hlv_ur)
    results["project capacity"].append(capacity)
    sim.env.cleanup_log_files(log_only=True)

              dinwoodie_CVOW-C_100pct_reduction-90 | 1.91 m
        dinwoodie_Ocean_Wind_1_100pct_reduction-90 | 3.79 m
            dinwoodie_fewer_techs_100pct_reduction | 1.71 m


In [5]:
# Save the results
# pickled dictionary format
with open(DINWOODIE / "outputs" / "results_dict_100pct_reduction_v0.4.1.yaml", "w") as f:
    yaml.dump(results, f, default_flow_style=False, sort_keys=False)

# dataframe/csv format
results_df = pd.DataFrame(results.values(), columns=columns, index=results.keys()).fillna(0)
results_df.to_csv(DINWOODIE / "outputs" / "results_data_100pct_reduction_v0.4.1.csv", index_label="result")

In [6]:
results_df

,Ocean_Wind_1_100pct_reduction-90,CVOW-C_100pct_reduction-90,fewer_techs_100pct_reduction
availability - time based,windfarm 0 0.94,windfarm 0 0.92,windfarm 0 0.94
availability - production based,windfarm 0 0.94,windfarm 0 0.91,windfarm 0 0.94
capacity factor - net,windfarm 0 0.56,windfarm 0 0.54,windfarm 0 0.45
capacity factor - gross,windfarm 0 0.59,windfarm 0 0.59,windfarm 0 0.47
power production,"54,064,860,732.32","120,255,159,001.10","9,030,836,695.00"
task completion rate,windfarm 0 1.00,windfarm 0 1.00,windfarm 0 1.00
annual direct O&M cost,24.59,40.84,13.24
annual vessel cost,13.79,18.01,9.78
ctv cost,1.92,1.88,1.85
fsv cost,0.16,0.32,0.16
